# Advanced Hybrid CNN+ViT - 5 New Ideas Training Pipeline

Bu notebook, 5 yeni mimari fikrini tek tek egitir, checkpoint ile devam eder, test setinde tum metrikleri hesaplar, en iyi modeli belirler ve gorsellestirmeleri uretir.

- Dataset: APTOS2019 (orijinal veri)
- Split: 80% train, 10% validation, 10% test
- Nihai degerlendirme: test seti

In [ ]:
# 1) Import Required Libraries
import os
import sys
import json
import warnings
from pathlib import Path
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Libraries imported')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Setup Configuration and Paths
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results_dir = Path('results/advanced_hybrid_cnn_vit')
results_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    'models/advanced_hybrid_models.py',
    'advanced_model_configs.py',
    'dataset_loader.py',
    'metrics.py'
]

print('Device:', device)
print('Results dir:', results_dir)
print('Checking required files:')
for f in required_files:
    ok = Path(f).exists()
    print(' -', f, '=>', 'OK' if ok else 'MISSING')
    if not ok:
        raise FileNotFoundError(f'Missing required file: {f}')

In [ ]:
# 3) Import Local Modules and List 5 Models
import importlib

for m in [
    'models.advanced_hybrid_models',
    'advanced_model_configs',
    'dataset_loader',
    'metrics'
]:
    if m in sys.modules:
        importlib.reload(sys.modules[m])

from models.advanced_hybrid_models import AdvancedHybridModel, ContrastiveLoss, PrototypicalLoss
from advanced_model_configs import get_advanced_model_config, list_advanced_models
from dataset_loader import get_data_loaders
from metrics import compute_metrics

models_list = list_advanced_models()
print('Available advanced models:', len(models_list))
for i, model_name in enumerate(models_list, 1):
    cfg = get_advanced_model_config(model_name)
    print(f'{i}. {cfg["name"]}: {cfg["description"]}')

In [ ]:
# 4) Load Dataset and Create DataLoaders (80-10-10)
# get_data_loaders() in this project already uses 80/10/10 default split internally.
batch_size = 16
train_loader, val_loader, test_loader, class_weights, split_data = get_data_loaders(
    dataset_path='APTOS2019',
    batch_size=batch_size
)

X_train, y_train, X_val, y_val, X_test, y_test = split_data
total = len(X_train) + len(X_val) + len(X_test)

print('Dataset loaded')
print(f'Train: {len(X_train)} ({len(X_train)/total*100:.1f}%)')
print(f'Val:   {len(X_val)} ({len(X_val)/total*100:.1f}%)')
print(f'Test:  {len(X_test)} ({len(X_test)/total*100:.1f}%)')
print('Batch size:', batch_size)

## 5) Define Advanced Trainer

Bu trainer su yetenekleri kapsar:
- checkpoint ile kaldigi yerden devam
- auxiliary loss destegi (contrastive/prototypical)
- early stopping (val QWK)
- en iyi modeli kaydetme
- test setinde final metrikler

In [ ]:
class AdvancedModelTrainer:
    def __init__(self, model_name: str, config: Dict, device: torch.device,
                 num_epochs: int = 100, learning_rate: float = 1e-4):
        self.model_name = model_name
        self.config = config
        self.device = device
        self.num_epochs = num_epochs

        self.model_dir = results_dir / model_name
        self.model_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_path = self.model_dir / 'checkpoint_current.pth'
        self.best_model_path = self.model_dir / 'best_model.pth'
        self.history_path = self.model_dir / 'training_history.json'
        self.metrics_path = self.model_dir / 'test_metrics.json'

        self.model = AdvancedHybridModel(num_classes=5, config=config).to(device)
        self.optimizer = optim.AdamW(self.model.parameters(), lr=learning_rate, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(self.optimizer, T_0=10, T_mult=2)

        if isinstance(class_weights, dict):
            cw = [class_weights.get(i, 1.0) for i in range(5)]
        else:
            cw = [1.0] * 5
        self.ce_loss = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=device))

        self.aux_loss_fns = {}
        aux_cfg = self.config.get('auxiliary_losses', {})
        for aux_name, aux_params in aux_cfg.items():
            aux_type = aux_params.get('type', '')
            if aux_type == 'contrastive':
                self.aux_loss_fns[aux_name] = (ContrastiveLoss(temperature=0.1), float(aux_params.get('weight', 0.3)))
            elif aux_type == 'prototypical':
                self.aux_loss_fns[aux_name] = (PrototypicalLoss(), float(aux_params.get('weight', 0.5)))

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_acc': [],
            'val_f1': [],
            'val_qwk': []
        }
        self.best_qwk = -1e9
        self.start_epoch = 0

        self._load_checkpoint()
        print(f'Trainer ready: {self.model_name}')

    def _load_checkpoint(self):
        if self.checkpoint_path.exists():
            ckpt = torch.load(self.checkpoint_path, map_location=self.device)
            self.model.load_state_dict(ckpt['model_state'])
            self.optimizer.load_state_dict(ckpt['optimizer_state'])
            self.scheduler.load_state_dict(ckpt['scheduler_state'])
            self.history = ckpt['history']
            self.best_qwk = ckpt['best_qwk']
            self.start_epoch = int(ckpt['epoch']) + 1
            print(f'Resumed from epoch {self.start_epoch}, best QWK={self.best_qwk:.4f}')

    def _calc_aux_losses(self, extra_outputs: Dict, labels_idx: torch.Tensor) -> torch.Tensor:
        aux_total = torch.tensor(0.0, device=self.device)
        for aux_name, (loss_fn, w) in self.aux_loss_fns.items():
            if aux_name == 'contrastive' and 'cnn_proj' in extra_outputs and 'vit_proj' in extra_outputs:
                aux_total = aux_total + w * loss_fn(extra_outputs['cnn_proj'], extra_outputs['vit_proj'], labels_idx)
            elif aux_name == 'prototypical' and 'final_feat' in extra_outputs and 'prototypes' in extra_outputs:
                aux_total = aux_total + w * loss_fn(extra_outputs['final_feat'], extra_outputs['prototypes'], labels_idx)
        return aux_total

    def train_epoch(self, loader: DataLoader) -> float:
        self.model.train()
        total = 0.0
        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device)
            labels_idx = labels.argmax(1) if labels.ndim > 1 else labels

            self.optimizer.zero_grad()
            logits, extra = self.model(images)

            loss_main = self.ce_loss(logits, labels_idx)
            loss_aux = self._calc_aux_losses(extra, labels_idx)
            loss = loss_main + loss_aux

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            total += float(loss.item())

        return total / max(1, len(loader))

    def validate(self, loader: DataLoader) -> Tuple[float, float, float, float]:
        self.model.eval()
        total = 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                labels_idx = labels.argmax(1) if labels.ndim > 1 else labels

                logits, _ = self.model(images)
                loss = self.ce_loss(logits, labels_idx)
                total += float(loss.item())

                preds = logits.argmax(1).detach().cpu().numpy()
                all_preds.extend(preds.tolist())
                all_labels.extend(labels_idx.detach().cpu().numpy().tolist())

        val_loss = total / max(1, len(loader))
        acc, f1, qwk = compute_metrics(all_labels, all_preds)
        return val_loss, float(acc), float(f1), float(qwk)

    def train(self, train_loader: DataLoader, val_loader: DataLoader):
        patience = 15
        patience_counter = 0

        for epoch in range(self.start_epoch, self.num_epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss, val_acc, val_f1, val_qwk = self.validate(val_loader)

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['val_f1'].append(val_f1)
            self.history['val_qwk'].append(val_qwk)

            ckpt = {
                'epoch': epoch,
                'model_state': self.model.state_dict(),
                'optimizer_state': self.optimizer.state_dict(),
                'scheduler_state': self.scheduler.state_dict(),
                'history': self.history,
                'best_qwk': self.best_qwk
            }
            torch.save(ckpt, self.checkpoint_path)

            if val_qwk > self.best_qwk:
                self.best_qwk = val_qwk
                torch.save(self.model.state_dict(), self.best_model_path)
                patience_counter = 0
            else:
                patience_counter += 1

            self.scheduler.step()

            print(f'Epoch {epoch+1:03d} | tr_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | val_qwk={val_qwk:.4f}')

            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

        with open(self.history_path, 'w', encoding='utf-8') as f:
            json.dump(self.history, f, indent=2)

    def evaluate_test(self, test_loader: DataLoader) -> Dict:
        self.model.load_state_dict(torch.load(self.best_model_path, map_location=self.device))
        self.model.eval()

        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                labels_idx = labels.argmax(1) if labels.ndim > 1 else labels

                logits, _ = self.model(images)
                preds = logits.argmax(1)

                all_preds.extend(preds.detach().cpu().numpy().tolist())
                all_labels.extend(labels_idx.detach().cpu().numpy().tolist())

        acc, f1, qwk = compute_metrics(all_labels, all_preds)
        cm = confusion_matrix(all_labels, all_preds)
        class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
        report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

        result = {
            'model_name': self.model_name,
            'accuracy': float(acc),
            'f1_macro': float(f1),
            'qwk': float(qwk),
            'best_val_qwk': float(self.best_qwk),
            'confusion_matrix': cm.tolist(),
            'classification_report': report
        }

        with open(self.metrics_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2)

        return result

print('AdvancedModelTrainer defined')

## 6) Run Training Loop for 5 Ideas

Bu hucre tum modelleri donguye alir. Baglanti koparsa checkpoint dosyasindan devam eder.

In [ ]:
all_results = {}
summary_rows = []

print('=' * 80)
print('ADVANCED HYBRID TRAINING - 5 MODELS')
print('=' * 80)

for idx, model_name in enumerate(list_advanced_models(), 1):
    print('\n' + '#' * 80)
    print(f'Model {idx}/5: {model_name}')
    print('#' * 80)

    cfg = get_advanced_model_config(model_name)
    try:
        trainer = AdvancedModelTrainer(
            model_name=model_name,
            config=cfg,
            device=device,
            num_epochs=100,
            learning_rate=1e-4
        )

        trainer.train(train_loader, val_loader)
        test_result = trainer.evaluate_test(test_loader)

        all_results[model_name] = test_result
        summary_rows.append({
            'Model': model_name,
            'Accuracy': test_result['accuracy'],
            'F1': test_result['f1_macro'],
            'QWK': test_result['qwk'],
            'Best_Val_QWK': test_result['best_val_qwk']
        })

        print(f"Test -> Acc: {test_result['accuracy']:.4f}, F1: {test_result['f1_macro']:.4f}, QWK: {test_result['qwk']:.4f}")

    except Exception as e:
        print('Error on', model_name, ':', str(e))
        all_results[model_name] = {'error': str(e)}

print('\nTraining loop finished')

In [ ]:
# 7) Summary Table (sorted by QWK)
if len(summary_rows) == 0:
    print('No successful run found yet.')
else:
    comparison_df = pd.DataFrame(summary_rows).sort_values('QWK', ascending=False).reset_index(drop=True)
    print(comparison_df.to_string(index=False))

    out_csv = results_dir / 'advanced_5ideas_comparison.csv'
    comparison_df.to_csv(out_csv, index=False)
    print('Saved:', out_csv)

In [ ]:
# 8) Performance Ranking Plots
if len(summary_rows) == 0:
    print('No results to plot.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Advanced 5 Ideas - Test Performance Ranking', fontsize=14, fontweight='bold')

    metrics_to_plot = ['Accuracy', 'F1', 'QWK']
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(comparison_df)))

    for ax, metric in zip(axes, metrics_to_plot):
        sdf = comparison_df.sort_values(metric, ascending=True)
        ax.barh(sdf['Model'], sdf[metric], color=colors)
        ax.set_xlabel(metric)
        ax.set_title(metric + ' Ranking')
        ax.grid(axis='x', alpha=0.3)

        for i, v in enumerate(sdf[metric]):
            ax.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=8)

    plt.tight_layout()
    rank_png = results_dir / '00_advanced_5ideas_ranking.png'
    plt.savefig(rank_png, dpi=150, bbox_inches='tight')
    print('Saved:', rank_png)
    plt.show()

In [ ]:
# 9) Training and Validation Loss Curves
model_names = list_advanced_models()
fig, axes = plt.subplots(1, len(model_names), figsize=(5 * len(model_names), 4), sharey=False)
if len(model_names) == 1:
    axes = [axes]

for ax, model_name in zip(axes, model_names):
    hp = results_dir / model_name / 'training_history.json'
    if hp.exists():
        with open(hp, 'r', encoding='utf-8') as f:
            hist = json.load(f)
        ax.plot(hist.get('train_loss', []), label='Train')
        ax.plot(hist.get('val_loss', []), label='Val')
    ax.set_title(model_name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
loss_png = results_dir / '01_advanced_5ideas_loss_curves.png'
plt.savefig(loss_png, dpi=150, bbox_inches='tight')
print('Saved:', loss_png)
plt.show()

In [ ]:
# 10) Validation Metrics Evolution (QWK and Accuracy)
model_names = list_advanced_models()
fig, axes = plt.subplots(1, len(model_names), figsize=(5 * len(model_names), 4), sharey=True)
if len(model_names) == 1:
    axes = [axes]

for ax, model_name in zip(axes, model_names):
    hp = results_dir / model_name / 'training_history.json'
    if hp.exists():
        with open(hp, 'r', encoding='utf-8') as f:
            hist = json.load(f)
        ax.plot(hist.get('val_qwk', []), label='Val QWK', marker='o', markersize=2)
        ax.plot(hist.get('val_acc', []), label='Val Acc', marker='s', markersize=2)
    ax.set_title(model_name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_ylim([0, 1])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
metric_png = results_dir / '02_advanced_5ideas_val_metrics.png'
plt.savefig(metric_png, dpi=150, bbox_inches='tight')
print('Saved:', metric_png)
plt.show()

In [ ]:
# 11) Confusion Matrix of Best Model (based on test QWK)
if len(summary_rows) == 0:
    print('No model result found.')
else:
    best_name = comparison_df.iloc[0]['Model']
    best_result = all_results[best_name]

    if 'confusion_matrix' in best_result:
        cm = np.array(best_result['confusion_matrix'])
        labels = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

        plt.figure(figsize=(7, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title(f'Best Model Confusion Matrix: {best_name}')
        plt.xlabel('Predicted')
        plt.ylabel('True')

        cm_png = results_dir / f'03_confusion_matrix_{best_name}.png'
        plt.tight_layout()
        plt.savefig(cm_png, dpi=150, bbox_inches='tight')
        print('Saved:', cm_png)
        plt.show()
    else:
        print('Best model does not contain confusion matrix data.')